# Типы карт × MCC (аналитика, без пересборки final_df)

Какие бренды карт есть в периметре эквайринга и в каком соотношении, в том числе **внутри MCC**.

| Поле | Откуда |
|---|---|
| Тип карты | `scd1_trx.c_fiid_iss` → `scd1_base24_fiids.c_fiid_desc` (эмитент) |
| MCC | `scd1_trx.n_mcc` (с операции) |
| Периметр | как секция 05 / `vd_acq_mcc_month.sql`: SA / S01 / RSHB / не `R` |

`final_df` и `tmp_shestopalov_acq_mcc_month` **не** трогаем.  
В SQL сразу агрегат `card_type × mcc` за один месяц — строки trx на клиент не выгружаем.

SQL: `sources/sql/vd_acq_card_type_mcc_month.sql`.

Новый kernel нормален. Нужен Impala (`tech.keytab` на `/home/jovyan`).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 80)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd()
OUT_DIR = DATA_DIR / 'qc_card_type_vs_mcc'
CKPT_DIR = OUT_DIR / 'checkpoints'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

period_start = '2026-01-01'
period_end_exclusive = '2026-09-01'
period_months = pd.date_range(
    period_start,
    pd.to_datetime(period_end_exclusive) - pd.Timedelta(days=1),
    freq='MS',
)

use_month_checkpoints = True
force_refetch_months = []  # ['2026-06'] или ['*']
MEM_LIMIT = '16g'
NOTEBOOK_REV = '2026-09-18-card-type-mcc-v1'

print('rev', NOTEBOOK_REV)
print('months', [m.strftime('%Y-%m') for m in period_months])
print('OUT_DIR', OUT_DIR)
print('checkpoints', use_month_checkpoints, 'force', force_refetch_months)


## 0) Helpers + Impala


In [ ]:
import re


def classify_brand(text):
    s = '' if pd.isna(text) else str(text).strip().lower()
    if not s or s in {'none', 'nan', 'null', 'unknown', 'empty'}:
        return 'empty'
    if re.search(r'union\s*pay|unionpay|юнион', s):
        return 'UnionPay'
    if re.search(r'\bmir\b|мир|nspk|нспк', s):
        return 'МИР'
    if re.search(r'master\s*card|mastercard|\bmaestro\b', s):
        return 'Mastercard'
    if re.search(r'\bvisa\b|виза', s):
        return 'Visa'
    if re.search(r'\bjcb\b', s):
        return 'JCB'
    if re.search(r'amex|american\s*express', s):
        return 'Amex'
    if re.search(r'\bsbp\b|сбп', s):
        return 'СБП'
    return 'other'


def _ckpt_path(label):
    return CKPT_DIR / f'card_type_mcc_{label}.parquet'


def _month_needs_fetch(label):
    if not use_month_checkpoints:
        return True
    if force_refetch_months == ['*'] or label in force_refetch_months:
        return True
    return not _ckpt_path(label).exists() and not _ckpt_path(label).with_suffix('.csv.gz').exists()


def _load_ckpt(label):
    pq = _ckpt_path(label)
    gz = pq.with_suffix('.csv.gz')
    if pq.exists():
        return pd.read_parquet(pq)
    return pd.read_csv(gz, compression='gzip')


def _save_ckpt(df, label):
    pq = _ckpt_path(label)
    try:
        df.to_parquet(pq, index=False)
        return pq
    except Exception as exc:
        gz = pq.with_suffix('.csv.gz')
        df.to_csv(gz, index=False, compression='gzip')
        print(f'  {label}: parquet fail ({type(exc).__name__}), saved {gz.name}')
        return gz


def sql_month(month_start, month_end_exclusive):
    return f'''
    WITH fiid_rshb AS (
      SELECT DISTINCT CAST(fa.c_fiid AS STRING) AS c_fiid
      FROM ods_alpha.scd1_base24_fiids fa
      WHERE COALESCE(CAST(fa.c_fiid_grp AS STRING), 'UNKNOWN') = 'RSHB'
    ),
    sa_agr AS (
      SELECT DISTINCT CAST(a.n_agr AS STRING) AS n_agr
      FROM ods_alpha.scd1_agreements a
      WHERE UPPER(TRIM(CAST(a.acq_class AS STRING))) = 'SA'
        AND a.abs_agr_id IS NOT NULL
    ),
    trx_base_raw AS (
      SELECT
        CAST(t.n_trx AS STRING) AS n_trx,
        CAST(t.n_mcc AS STRING) AS mcc,
        CAST(t.c_fiid_iss AS STRING) AS c_fiid_iss,
        CAST(t.n_amt_src AS DOUBLE) AS n_amt_src
      FROM ods_alpha.scd1_trx t
      JOIN fiid_rshb fr
        ON fr.c_fiid = CAST(t.c_fiid_acq AS STRING)
      WHERE CAST(t.d_trx_orig AS TIMESTAMP) >= CAST('{month_start}' AS TIMESTAMP)
        AND CAST(t.d_trx_orig AS TIMESTAMP) < CAST('{month_end_exclusive}' AS TIMESTAMP)
        AND t.c_nter IS NOT NULL
        AND COALESCE(t.ods_deleted_flg, '0') <> '1'
        AND t.c_trx_class = 'SA'
        AND t.c_trx_type = 'S01'
        AND COALESCE(t.cf_trx_stat, '') <> 'R'
    ),
    trx_base AS (
      SELECT
        n_trx,
        MAX(mcc) AS mcc,
        MAX(c_fiid_iss) AS c_fiid_iss,
        MAX(n_amt_src) AS n_amt_src
      FROM trx_base_raw
      GROUP BY n_trx
    ),
    ta AS (
      SELECT
        CAST(a.n_trx AS STRING) AS n_trx,
        MAX(COALESCE(CAST(a.n_amt_tax AS DOUBLE), 0.0)) AS n_amt_tax
      FROM ods_alpha.scd1_trx_acq a
      JOIN trx_base tb ON tb.n_trx = CAST(a.n_trx AS STRING)
      JOIN sa_agr ss ON ss.n_agr = CAST(a.n_agr AS STRING)
      GROUP BY CAST(a.n_trx AS STRING)
    )
    SELECT
      COALESCE(NULLIF(TRIM(CAST(fi.c_fiid_desc AS STRING)), ''), 'empty') AS card_type,
      COALESCE(NULLIF(TRIM(tb.mcc), ''), 'empty') AS mcc,
      COUNT(DISTINCT tb.n_trx) AS trx_cnt,
      SUM(tb.n_amt_src) AS trx_sum,
      SUM(ta.n_amt_tax) AS commission_from_ops
    FROM trx_base tb
    JOIN ta ON ta.n_trx = tb.n_trx
    LEFT JOIN ods_alpha.scd1_base24_fiids fi
      ON CAST(fi.c_fiid AS STRING) = tb.c_fiid_iss
    GROUP BY 1, 2
    '''


if 'imp' in globals() and imp is not None:
    print('Reuse existing Impala')
else:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')


## 1) Помесячная выгрузка `card_type × mcc`

Один запрос на весь Jan–Aug тяжёлый. Цикл по месяцу, результат — сотни/тысячи строк, не миллиарды.


In [ ]:
EMPTY_COLS = ['card_type', 'mcc', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'report_month']
parts = []
n_hit = n_fetch = 0

for m in period_months:
    m_start = m.strftime('%Y-%m-%d')
    m_end_excl = (m + pd.offsets.MonthBegin(1)).strftime('%Y-%m-%d')
    label = m.strftime('%Y-%m')

    if not _month_needs_fetch(label):
        part = _load_ckpt(label)
        n_hit += 1
        print(f'  {label}: checkpoint, rows={len(part):,}')
    else:
        print(f'  {label}: fetch Impala...', flush=True)
        with imp:
            try:
                imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
            except Exception:
                pass
            part = imp.fetch(sql_month(m_start, m_end_excl))
        n_fetch += 1
        if part is None or len(part) == 0:
            part = pd.DataFrame(columns=EMPTY_COLS)
            print(f'  {label}: empty')
        else:
            part = part.copy()
            part['report_month'] = label
            print(f'  {label}: card_type×mcc rows={len(part):,}')
        saved = _save_ckpt(part, label)
        print(f'  {label}: saved {saved.name}')

    if part is None or len(part) == 0:
        continue
    part = part.copy()
    if 'report_month' not in part.columns:
        part['report_month'] = label
    parts.append(part)

print(f'checkpoint hits={n_hit}, Impala fetches={n_fetch}')

if parts:
    raw = pd.concat(parts, ignore_index=True)
else:
    raw = pd.DataFrame(columns=EMPTY_COLS)

raw['report_month'] = raw['report_month'].astype(str).str[:7]
raw['card_type'] = raw['card_type'].astype(str).str.strip()
raw['mcc'] = raw['mcc'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
for c in ['trx_cnt', 'trx_sum', 'commission_from_ops']:
    raw[c] = pd.to_numeric(raw[c], errors='coerce').fillna(0.0)
raw['brand'] = raw['card_type'].map(classify_brand)

print(f'rows={len(raw):,} months={sorted(raw["report_month"].dropna().unique().tolist())}')
print('brands:', sorted(raw['brand'].unique().tolist()))
display(raw.head(15))


## 2) Какие типы карт и в каком соотношении

Доли по числу операций, обороту и комиссии. Сначала всё окно, затем помесячно.


In [ ]:
def add_shares(df, group_cols, metrics=('trx_cnt', 'trx_sum', 'commission_from_ops')):
    out = df.copy()
    for m in metrics:
        tot = out.groupby(group_cols)[m].transform('sum') if group_cols else out[m].sum()
        out[f'{m}_share_pct'] = np.where(tot == 0, np.nan, 100.0 * out[m] / tot)
    return out


brand_all = (
    raw.groupby('brand', dropna=False)
    .agg(
        trx_cnt=('trx_cnt', 'sum'),
        trx_sum=('trx_sum', 'sum'),
        commission_from_ops=('commission_from_ops', 'sum'),
        card_type_n=('card_type', 'nunique'),
        mcc_n=('mcc', 'nunique'),
    )
    .reset_index()
)
brand_all = add_shares(brand_all, [])
brand_all = brand_all.sort_values('trx_sum', ascending=False)

print('=== смесь брендов за Jan–Aug ===')
display(brand_all)

brand_month = (
    raw.groupby(['report_month', 'brand'], dropna=False)
    .agg(
        trx_cnt=('trx_cnt', 'sum'),
        trx_sum=('trx_sum', 'sum'),
        commission_from_ops=('commission_from_ops', 'sum'),
    )
    .reset_index()
)
brand_month = add_shares(brand_month, ['report_month'])
brand_month = brand_month.sort_values(['report_month', 'trx_sum'], ascending=[True, False])

print('=== доли брендов по месяцам (оборот) ===')
pivot_sum = brand_month.pivot(index='report_month', columns='brand', values='trx_sum_share_pct')
display(pivot_sum)
print('=== доли по числу операций ===')
display(brand_month.pivot(index='report_month', columns='brand', values='trx_cnt_share_pct'))

print('=== сырой card_type (топ 30 по обороту) ===')
desc_all = (
    raw.groupby(['brand', 'card_type'], dropna=False)
    .agg(trx_cnt=('trx_cnt', 'sum'), trx_sum=('trx_sum', 'sum'))
    .reset_index()
    .sort_values('trx_sum', ascending=False)
)
display(desc_all.head(30))


## 3) Кросс brand × MCC

- `share_in_mcc_pct` — доля бренда внутри MCC (сумма по брендам в одном MCC ≈ 100%).
- `share_in_brand_pct` — доля MCC внутри бренда.


In [ ]:
cross = (
    raw.groupby(['brand', 'mcc'], dropna=False)
    .agg(
        trx_cnt=('trx_cnt', 'sum'),
        trx_sum=('trx_sum', 'sum'),
        commission_from_ops=('commission_from_ops', 'sum'),
    )
    .reset_index()
)
mcc_tot = cross.groupby('mcc')[['trx_cnt', 'trx_sum']].transform('sum')
brand_tot = cross.groupby('brand')[['trx_cnt', 'trx_sum']].transform('sum')
cross['share_in_mcc_pct'] = np.where(mcc_tot['trx_sum'] == 0, np.nan, 100.0 * cross['trx_sum'] / mcc_tot['trx_sum'])
cross['share_in_brand_pct'] = np.where(brand_tot['trx_sum'] == 0, np.nan, 100.0 * cross['trx_sum'] / brand_tot['trx_sum'])
cross = cross.sort_values(['mcc', 'trx_sum'], ascending=[True, False])

print(f'cross rows={len(cross):,} unique MCC={cross["mcc"].nunique():,}')
print('=== топ 25 пар brand×MCC по обороту ===')
display(cross.sort_values('trx_sum', ascending=False).head(25))

print('=== топ-8 MCC внутри каждого бренда ===')
top_per_brand = (
    cross.sort_values(['brand', 'trx_sum'], ascending=[True, False])
    .groupby('brand', as_index=False)
    .head(8)
)
display(top_per_brand)

print('=== для крупных MCC: какой бренд держит оборот ===')
big_mcc = (
    cross.groupby('mcc', as_index=False)['trx_sum'].sum()
    .sort_values('trx_sum', ascending=False)
    .head(15)['mcc']
)
display(
    cross[cross['mcc'].isin(big_mcc)]
    .sort_values(['mcc', 'trx_sum'], ascending=[True, False])
)


## 4) Сохранение CSV


In [ ]:
raw_path = OUT_DIR / 'card_type_mcc_month_2026_01_2026_08.csv'
brand_all_path = OUT_DIR / 'card_type_brand_mix.csv'
brand_month_path = OUT_DIR / 'card_type_brand_by_month.csv'
cross_path = OUT_DIR / 'card_type_brand_x_mcc.csv'

raw.to_csv(raw_path, index=False, encoding='utf-8-sig')
brand_all.to_csv(brand_all_path, index=False, encoding='utf-8-sig')
brand_month.to_csv(brand_month_path, index=False, encoding='utf-8-sig')
cross.to_csv(cross_path, index=False, encoding='utf-8-sig')

print('saved')
print(' ', raw_path)
print(' ', brand_all_path)
print(' ', brand_month_path)
print(' ', cross_path)
print()
print('VERDICT: смотрите brand_all (смесь) и cross (привязка к MCC).')
print('В таблицу MCC и final_df ничего не писали.')
